# MiniGPT on a faster engine: follow along

This notebook goes with my post [MiniGPT (Part 4)](https://haddley.github.io/posts/minigpt3/). It rebuilds Part 3's machine in Apple's MLX, checks that it is the same machine, trains it, races MLX against PyTorch, and writes a story.

**It needs a Mac with Apple Silicon**, because MLX only runs there. Open it in Jupyter from a clone of [minigpt-series](https://github.com/Haddley/minigpt-series), with the packages from its `requirements.txt` installed.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("."))
from notebook_setup import setup
run = setup("part4-mlx")

working in part4-mlx


## 1. The same machine, in two engines

Build Part 3's PyTorch machine and the MLX one with the same 8,192 token cards, and count their numbers.

In [2]:
import importlib.util
import mlx.core as mx
from mlx.utils import tree_flatten
from model_mlx import MiniGPT as MLXMiniGPT

spec = importlib.util.spec_from_file_location("torch_model", "../part3-tokenisers/model.py")
torch_model = importlib.util.module_from_spec(spec); spec.loader.exec_module(torch_model)

pt = torch_model.MiniGPT(8192)
mlx_model = MLXMiniGPT(8192)
mx.eval(mlx_model.parameters())
print("PyTorch:", f"{sum(p.numel() for p in pt.parameters()):,}", "numbers")
print("MLX:    ", f"{sum(v.size for _, v in tree_flatten(mlx_model.parameters())):,}", "numbers")

PyTorch: 13,882,368 numbers
MLX:     13,882,368 numbers


## 2. Lazy evaluation

MLX writes calculations down and only runs them when a result is needed. Building a big multiplication is instant; asking for its result is where the work happens.

In [3]:
import time
a = mx.random.normal((4096, 4096))
b = mx.random.normal((4096, 4096))
mx.eval(a, b)

t = time.time(); c = a @ b @ a @ b;  print(f"writing the steps down: {(time.time() - t) * 1000:8.2f} ms")
t = time.time(); mx.eval(c);         print(f"calculating them:       {(time.time() - t) * 1000:8.2f} ms")

writing the steps down:     0.05 ms
calculating them:          62.76 ms


## 3. Train the MLX machine

3,000 training steps on Part 3's stories, checking bits per byte every 300 steps. On my Mac Studio, 3,000 steps took about 8 minutes.

In [4]:
run("train_mlx.py", "--tokenizer", "bpe8k", "--iters", "3000", "--eval-interval", "300")

tokenizer bpe8k  vocab 8,192  train tokens 4,928,114  tokens/byte 0.2436
parameters 13,882,368
step     0  train 9.5896  val 9.5847  val bpb 3.3685
step   300  train 3.1420  val 3.1922  val bpb 1.1218
step   600  train 2.5907  val 2.6721  val bpb 0.9391
step   900  train 2.3439  val 2.4340  val bpb 0.8554
step  1200  train 2.1571  val 2.2990  val bpb 0.8080
step  1500  train 2.0407  val 2.1805  val bpb 0.7663
step  1800  train 1.9492  val 2.1040  val bpb 0.7394
step  2100  train 1.9143  val 2.0580  val bpb 0.7232
step  2400  train 1.8421  val 2.0177  val bpb 0.7091
step  2700  train 1.8091  val 1.9876  val bpb 0.6985
step  3000  train 1.7665  val 1.9591  val bpb 0.6885
done in 7.71 min  best val 1.9591  53,103 tok/s  peak 4.02 GB



## 4. The race

The same 200 timed training steps in each engine, each in its own fresh program, so the memory figures stay clean.

In [5]:
for framework in ("torch", "mlx-nocompile", "mlx"):
    run("bench.py", "--framework", framework)

PyTorch-MPS           48,580 tok/s   peak  4.60 GB   (vocab 8,192, batch 32, block 256)



MLX (no compile)      46,558 tok/s   peak  3.79 GB   (vocab 8,192, batch 32, block 256)



MLX                   58,149 tok/s   peak  3.90 GB   (vocab 8,192, batch 32, block 256)



## 5. What it writes

In [6]:
run("generate_mlx.py", "--prompt", "Once upon a time", "--max-new-tokens", "150")

Once upon a time, there was a little boy named Tim. Tim had a big, dirty sock. He liked to play with his friends every day.
One sunny day, Tim and his friends wanted to play a game with the smelly sock. But Tim was too persistent. He asked his friends, the little birds, and the squirrels to help them.
They all played the dirty sock. The smelly sock was clean and pretty again. The wind made them feel better. Tim and his friends watched and wave. They were very happy.

Once upon a time, there was a little girl named Lily. She loved to play with her toys and have fun. One day, her mom told her that they were all having fun. Lily was so happy

